# LeetCode 478 — Generate Random Point in a Circle bằng Python

Link đề: https://leetcode.com/problems/generate-random-point-in-a-circle/ · Độ khó: **Medium** · Chủ đề: Math, Geometry (hình học), Randomized (ngẫu nhiên), Rejection Sampling (lấy mẫu loại bỏ)

Skeleton LeetCode:

```python
class Solution:
    def __init__(self, radius: float, x_center: float, y_center: float):
        ...

    def randPoint(self) -> List[float]:
        ...
```

Ghi chú: bài này **không** phải Premium. Đây là bài **thiết kế lớp có tính ngẫu nhiên**: không có "đáp án cứng" để so sánh, nên notebook kiểm tra bằng **tính chất** (điểm có nằm trong hình tròn không) và **thống kê** (phân bố có **đều** không) với **seed cố định** để kết quả tái lập được.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: sinh một điểm ngẫu nhiên **phân bố đều theo diện tích** trong hình tròn (xác suất rơi vào một vùng tỉ lệ với diện tích vùng đó); điểm trên đường tròn cũng tính là "trong hình tròn".
2. Dùng các công cụ Python/toán: `random.random`, `random.uniform`, `random.seed` (tái lập kết quả), `math.sqrt`, `math.cos`/`math.sin`, tọa độ cực `(r, θ)` → tọa độ Descartes `(x, y)`, và tự dựng một **bộ sinh số ngẫu nhiên** nhỏ có seed.
3. Hiểu vì sao "chọn bán kính đều + chọn góc đều" là **SAI** (điểm dồn về tâm) và vì sao phải lấy **căn bậc hai**: diện tích của vòng nhỏ tỉ lệ với `r^2`, nên `r^2` mới là đại lượng cần phân bố đều.
4. Cài đặt nhiều cách: **baseline sai (bán kính đều) → lấy mẫu loại bỏ trong hình vuông bao → tọa độ cực với căn bậc hai → hướng ngẫu nhiên từ phân phối chuẩn → mẹo `max(U1, U2)`**, và so sánh chi phí.
5. Biết **kiểm thử thống kê**: kiểm tra điểm nằm trong hình tròn, kiểm tra phân bố đều bằng thống kê **khi-bình-phương** (chi-square) trên các **vòng diện tích bằng nhau** và các **cung góc bằng nhau**, với seed cố định.

## 2. Đề bài & ví dụ

Cho **bán kính** và **vị trí tâm** của một hình tròn, hãy cài hàm `randPoint` sinh ra một **điểm ngẫu nhiên đều** trong hình tròn. Điểm nằm trên đường tròn cũng được coi là nằm trong hình tròn. Kết quả trả về là mảng `[x, y]`.

Cài đặt lớp `Solution`:

- `Solution(double radius, double x_center, double y_center)` khởi tạo đối tượng với bán kính `radius` và tâm `(x_center, y_center)`.
- `randPoint()` trả về một điểm ngẫu nhiên trong hình tròn dưới dạng `[x, y]`.

**Ví dụ:**

```
Input:  ["Solution", "randPoint", "randPoint", "randPoint"]
        [[1.0, 0.0, 0.0], [], [], []]
Output: [null, [-0.02493, -0.38077], [0.82314, 0.38945], [0.36572, 0.17248]]
```

(Ví dụ gốc: đối chiếu lại đề gốc nếu cần; vì kết quả là **ngẫu nhiên**, mỗi lần chạy sẽ cho các điểm khác nhau; chỉ cần các điểm **nằm trong hình tròn** và **phân bố đều**.)

**Ràng buộc:**

- `0 < radius <= 10^8`
- `-10^7 <= x_center, y_center <= 10^7`
- Tối đa `3 * 10^4` lần gọi `randPoint`.

Vì kết quả ngẫu nhiên, ta **không** so sánh với đáp án cứng. Ta kiểm tra: (1) mọi điểm nằm trong hình tròn; (2) phân bố **đều** bằng thống kê (mục 12).

In [1]:
import math
import random
import time
from typing import List

# Bộ chạy chuỗi lệnh mô phỏng cách LeetCode gọi: ops là tên lớp/method, args là tham số tương ứng
def chay(cls, ops, args):
    obj = None
    ket_qua = []
    for op, a in zip(ops, args):
        if op == "Solution":
            obj = cls(*a)
            ket_qua.append(None)
        else:
            ket_qua.append(getattr(obj, op)(*a))
    return ket_qua

ops = ["Solution", "randPoint", "randPoint", "randPoint"]
args = [[1.0, 0.0, 0.0], [], [], []]
print("ops :", ops)
print("args:", args)

ops : ['Solution', 'randPoint', 'randPoint', 'randPoint']
args: [[1.0, 0.0, 0.0], [], [], []]


## 3. Kiến thức nền

### 3.1. Số ngẫu nhiên và `seed`: cùng seed thì cùng chuỗi

`random.random()` trả về số thực **ngẫu nhiên đều** trong `[0, 1)`. Máy tính dùng một **bộ sinh giả ngẫu nhiên** (pseudo-random): bắt đầu từ một **seed** (hạt giống), chuỗi số sinh ra hoàn toàn **xác định**. `random.seed(s)` đặt seed; đặt lại cùng seed cho lại cùng chuỗi. Nhờ đó test ngẫu nhiên **tái lập** được.

In [2]:
random.seed(7)
a = [round(random.random(), 4) for _ in range(3)]
random.seed(7)
b = [round(random.random(), 4) for _ in range(3)]
print(a, b, a == b)                  # cùng seed -> cùng chuỗi
random.seed(8)
print([round(random.random(), 4) for _ in range(3)])   # seed khác -> chuỗi khác

[0.3238, 0.1508, 0.6509] [0.3238, 0.1508, 0.6509] True
[0.2267, 0.9623, 0.1263]


### 3.2. `random.uniform(a, b)` và `random.Random(seed)`

`random.uniform(a, b)` trả về số thực đều trong `[a, b]`; thực chất là `a + (b - a) * random.random()`. Ngoài bộ sinh toàn cục, ta có thể tạo **một bộ sinh riêng** `rng = random.Random(seed)` để không ảnh hưởng nơi khác.

In [3]:
rng = random.Random(123)
print(rng.uniform(-2, 2))
print(rng.uniform(-2, 2))
rng2 = random.Random(123)
x = rng2.random()
print(-2 + 4 * x)                    # trùng giá trị uniform(-2, 2) đầu tiên ở trên

-1.7905456045962227
-1.6512532899094707
-1.7905456045962227


### 3.3. Tự dựng bộ sinh số ngẫu nhiên có seed (LCG)

Chỉ cần nhớ: thuật toán của ta **không phụ thuộc** bộ sinh cụ thể; nó chỉ cần một nguồn số đều trong `[0, 1)`. Để thấy điều đó, ta tự viết một **bộ sinh đồng dư tuyến tính (LCG)** 64 bit: `state = (a * state + c) mod 2^64`, rồi lấy 53 bit cao chia cho `2^53` để ra số thực trong `[0, 1)`. Các hằng số dưới đây là của Knuth (MMIX).

In [4]:
class LCG:
    """Bộ sinh giả ngẫu nhiên đơn giản có seed: .random() cho số đều trong [0, 1)."""
    def __init__(self, seed):
        self.state = seed & (2**64 - 1)

    def random(self):
        self.state = (6364136223846793005 * self.state + 1442695040888963407) & (2**64 - 1)
        return (self.state >> 11) / (1 << 53)           # lấy 53 bit cao -> [0, 1)

g = LCG(2024)
print([round(g.random(), 4) for _ in range(5)])
g = LCG(2024)
print([round(g.random(), 4) for _ in range(5)])          # cùng seed -> cùng chuỗi
xs = [LCG(1).random() for _ in range(1)]
g = LCG(99)
m = sum(g.random() for _ in range(100000)) / 100000
print("trung binh 100000 so:", round(m, 4), "(ky vong 0.5)")

[0.3593, 0.3041, 0.7103, 0.2551, 0.9113]
[0.3593, 0.3041, 0.7103, 0.2551, 0.9113]
trung binh 100000 so: 0.5019 (ky vong 0.5)


### 3.4. Tọa độ cực → tọa độ Descartes

Một điểm cách tâm `(xc, yc)` khoảng `r` theo hướng góc `θ` (radian) có tọa độ `x = xc + r cos θ`, `y = yc + r sin θ`. `math.cos`/`math.sin` nhận **radian**; một vòng tròn đầy là `2π`.

In [5]:
xc, yc = 1.0, 2.0
for theta_do in (0, 90, 180, 270):
    theta = math.radians(theta_do)
    r = 3.0
    x = xc + r * math.cos(theta)
    y = yc + r * math.sin(theta)
    print(f"goc {theta_do:3} do: ({x:6.2f}, {y:6.2f})  khoang cach toi tam = {math.hypot(x - xc, y - yc):.2f}")

goc   0 do: (  4.00,   2.00)  khoang cach toi tam = 3.00
goc  90 do: (  1.00,   5.00)  khoang cach toi tam = 3.00
goc 180 do: ( -2.00,   2.00)  khoang cach toi tam = 3.00
goc 270 do: (  1.00,  -1.00)  khoang cach toi tam = 3.00


### 3.5. Diện tích và xác suất: "đều" nghĩa là gì

"Phân bố đều trong hình tròn" nghĩa là xác suất điểm rơi vào một vùng **tỉ lệ với diện tích** vùng đó. Hình tròn bán kính `R` có diện tích `π R^2`; hình tròn nhỏ đồng tâm bán kính `R/2` có diện tích `π R^2 / 4`, nên xác suất rơi vào hình tròn nhỏ phải là `1/4`, **không** phải `1/2`. Đây là chìa khóa của bài.

In [6]:
R = 1.0
for r in (0.25, 0.5, 0.75, 1.0):
    print(f"P(khoang cach <= {r}) = dien tich ti le = {(math.pi * r**2) / (math.pi * R**2):.4f}")

P(khoang cach <= 0.25) = dien tich ti le = 0.0625
P(khoang cach <= 0.5) = dien tich ti le = 0.2500
P(khoang cach <= 0.75) = dien tich ti le = 0.5625
P(khoang cach <= 1.0) = dien tich ti le = 1.0000


### 3.6. Khi-bình-phương (chi-square): đo độ lệch so với "đều"

Chia vùng thành `k` ô có **diện tích bằng nhau**; với `N` điểm đều, kỳ vọng mỗi ô có `N/k` điểm. Thống kê `χ² = Σ (quan sát - kỳ vọng)² / kỳ vọng`. Nếu phân bố đúng là đều, `χ²` xấp xỉ phân phối khi-bình-phương với `k - 1` bậc tự do (trung bình `k - 1`). Ta dùng **ngưỡng ở mức ý nghĩa 0,001**: `k = 10` (9 bậc tự do): `27.877`; `k = 8` (7 bậc tự do): `24.322`. Vượt ngưỡng = phân bố **không đều** (bằng chứng rất mạnh). Với seed cố định, kết quả luôn như nhau mỗi lần chạy.

In [7]:
def chi_binh_phuong(dem, ky_vong):
    return sum((o - ky_vong) ** 2 / ky_vong for o in dem)

# Ví dụ: 4 ô, N = 400 -> kỳ vọng 100 mỗi ô
print(chi_binh_phuong([100, 100, 100, 100], 100))        # 0.0  (đều tuyệt đối)
print(chi_binh_phuong([95, 108, 97, 100], 100))           # nhỏ: lệch bình thường
print(chi_binh_phuong([250, 50, 50, 50], 100))            # rất lớn: không đều

0.0
0.98
300.0


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

**Ý tưởng đầu tiên (hấp dẫn nhưng sai):** chọn bán kính `r` đều trong `[0, R]` và góc `θ` đều trong `[0, 2π)`. Nhìn có vẻ "đều" nhưng hãy **tính tay** xác suất rơi vào hình tròn nhỏ `r <= R/2`:

- Theo cách này `r` đều trong `[0, R]` nên `P(r <= R/2) = 1/2`.
- Nhưng diện tích hình tròn bán kính `R/2` chỉ bằng **1/4** diện tích cả hình tròn: một phân bố đều theo diện tích đòi hỏi `P = 1/4`.

Vậy cách đó dồn quá nhiều điểm về gần tâm (điểm đặc ở giữa, thưa ở rìa).

**Hướng sửa (trực giác):** để `P(r <= t R) = t^2` (tỉ lệ diện tích). Nếu ta lấy `U` đều trong `[0, 1)` và đặt `r = R sqrt(U)` thì `P(r <= t R) = P(sqrt(U) <= t) = P(U <= t^2) = t^2`. Đúng như cần.

**Hướng khác (lấy mẫu loại bỏ):** hình tròn nằm trong hình vuông cạnh `2R`. Chọn điểm **đều trong hình vuông** (dễ: hai số đều độc lập), nếu rơi **trong** hình tròn thì nhận, nếu **ngoài** thì bỏ và chọn lại. Vì hình vuông đều nên các điểm được nhận cũng đều trên hình tròn. Xác suất nhận mỗi lần `= π R^2 / (2R)^2 = π / 4 ≈ 0,785`, nên trung bình cần `4 / π ≈ 1,273` lần thử.

In [8]:
# Kiểm tra tính tay bằng máy: P(r <= R/2) theo hai cách chọn bán kính
random.seed(1)
N = 100000
R = 1.0
cach_sai = sum(1 for _ in range(N) if random.uniform(0, R) <= R / 2) / N
cach_dung = sum(1 for _ in range(N) if R * math.sqrt(random.random()) <= R / 2) / N
print(f"ban kinh deu     : P(r <= R/2) ~ {cach_sai:.3f}  (can 0.25 neu deu theo dien tich)")
print(f"ban kinh sqrt(U) : P(r <= R/2) ~ {cach_dung:.3f}")

ban kinh deu     : P(r <= R/2) ~ 0.500  (can 0.25 neu deu theo dien tich)
ban kinh sqrt(U) : P(r <= R/2) ~ 0.250


## 5. Cách 1 — Brute Force / baseline: bán kính đều + góc đều (cách ngây thơ, SAI)

**Ý tưởng:** `r = random.uniform(0, R)`, `θ = random.uniform(0, 2π)`, rồi đổi sang tọa độ Descartes cộng tâm. Code ngắn, điểm luôn **nằm trong hình tròn**, nhưng phân bố **không đều theo diện tích**.

Ta cài như một lớp `SolNaive` có cùng giao diện LeetCode để thử bằng bộ chạy `chay` và so với cách đúng.

In [9]:
class SolNaive:
    def __init__(self, radius: float, x_center: float, y_center: float):
        self.radius = radius
        self.x_center = x_center
        self.y_center = y_center

    def randPoint(self) -> List[float]:
        r = random.uniform(0, self.radius)                  # SAI: bán kính đều
        theta = random.uniform(0, 2 * math.pi)
        return [self.x_center + r * math.cos(theta), self.y_center + r * math.sin(theta)]

random.seed(478)
out = chay(SolNaive, ops, args)
print(out)

[None, [0.4998675866585367, -0.8244517343777135], [0.11401074297289954, 0.015442788753587998], [0.3571394418108456, 0.15888449575036542]]


**Độ phức tạp:** Time `O(1)` mỗi lần gọi, Space `O(1)`.

**Nút thắt:** sai về **phân bố**: điểm dồn về tâm. Ta cần công cụ **đo** để thấy lỗi này, vì nhìn vài điểm không thấy gì. Dưới đây là hàm vẽ **bản đồ mật độ** bằng ký tự ASCII (ô càng đậm nghĩa là càng nhiều điểm) và hàm tính `χ²` theo vòng diện tích bằng nhau.

In [10]:
def trong_hinh_tron(p, R, xc, yc):
    """True nếu điểm p nằm trong (hoặc trên) hình tròn, cho phép sai số làm tròn thập phân rất nhỏ."""
    sai_so = 1e-9 * max(1.0, abs(xc), abs(yc), R)
    return math.hypot(p[0] - xc, p[1] - yc) <= R + sai_so

def ve_mat_do(diem, R, xc, yc, k=21):
    """Vẽ bản đồ mật độ k x k trong hình vuông bao quanh hình tròn."""
    luoi = [[0] * k for _ in range(k)]
    for x, y in diem:
        i = min(k - 1, int((y - (yc - R)) / (2 * R) * k))
        j = min(k - 1, int((x - (xc - R)) / (2 * R) * k))
        luoi[i][j] += 1
    lon_nhat = max(max(h) for h in luoi) or 1
    bang = " .:-=+*#%@"
    for h in reversed(luoi):
        print("".join(bang[min(len(bang) - 1, int(c / lon_nhat * (len(bang) - 1) + 0.5))] * 2 for c in h))

random.seed(478)
diem_sai = [SolNaive(1.0, 0.0, 0.0).randPoint() for _ in range(6000)]
assert all(trong_hinh_tron(p, 1.0, 0.0, 0.0) for p in diem_sai)       # luôn trong hình tròn
print("Ban do mat do cua cach NGAY THO (6000 diem): tam rat dam, vien rat nhat")
ve_mat_do(diem_sai, 1.0, 0.0, 0.0)

Ban do mat do cua cach NGAY THO (6000 diem): tam rat dam, vien rat nhat
                                          
                                          
                                          
                                          
                                          
                  ..    ....              
          ..      ....  ......            
              ....  ........  ..          
        ....  ............  ..            
              ....::::::........          
          ..  ....--@@::....  ..          
              ....::::::......  ..        
      ......  ......::........  ..        
            ..  ..............            
              ..............              
                ..    ....                
                  ..                      
                        ..                
                                          
                                          
                                          


In [11]:
def chi2_vong(diem, R, xc, yc, k=10):
    """Chia hình tròn thành k vòng DIỆN TÍCH BẰNG NHAU: chỉ số vòng = floor(k * (d/R)^2)."""
    dem = [0] * k
    for x, y in diem:
        d2 = ((x - xc) ** 2 + (y - yc) ** 2) / (R * R)
        dem[min(k - 1, int(k * d2))] += 1
    return chi_binh_phuong(dem, len(diem) / k), dem

chi2, dem = chi2_vong(diem_sai, 1.0, 0.0, 0.0)
print("so diem moi vong (tam -> ria):", dem)
print("chi2 =", round(chi2, 1), " (nguong 27.877 -> khong deu!)")

so diem moi vong (tam -> ria): [1865, 742, 641, 510, 445, 375, 369, 377, 356, 320]
chi2 = 3243.1  (nguong 27.877 -> khong deu!)


Ta thấy vòng trong cùng chứa gần `31%` số điểm thay vì `10%`: **không đều**. Cách đo `χ²` theo vòng diện tích bằng nhau chính là công cụ kiểm thử sẽ dùng ở mục 12.

## 6. Ý tưởng thuật toán / bước đột phá

Có hai ý tưởng, đều dẫn tới phân bố đều theo diện tích:

**(a) Lấy mẫu loại bỏ (rejection sampling).** Không cần công thức: chọn điểm đều trong hình vuông bao `[xc - R, xc + R] × [yc - R, yc + R]`, giữ lại nếu thuộc hình tròn. Đúng vì phân bố có điều kiện của một phân bố đều trên một tập con cũng là phân bố đều trên tập con đó.

**(b) Biến đổi nghịch (inverse transform) trong tọa độ cực.** Muốn `P(r <= t R) = t^2`, đặt `r = R sqrt(U)` với `U` đều trong `[0, 1)`. Góc `θ` đều trong `[0, 2π)` và **độc lập** với `r` (vì hình tròn đối xứng quay).

**Biến thể của (b):** (i) lấy hướng ngẫu nhiên bằng cách chuẩn hóa vector hai số theo phân phối chuẩn (không dùng `sin`/`cos`); (ii) thay `sqrt(U)` bằng `max(U1, U2)` vì `P(max(U1, U2) <= t) = t * t = t^2`, cũng cho đúng phân phối cần.

## 7. Các cách tiếp theo

### Cách 2 — Lấy mẫu loại bỏ trong hình vuông bao

**Giải thích từng đoạn:**

1. Lặp vô hạn: `x = random.uniform(xc - R, xc + R)`, `y = random.uniform(yc - R, yc + R)`.
2. Nếu `(x - xc)^2 + (y - yc)^2 <= R^2` → trả `[x, y]`; ngược lại lặp lại.

Số lần thử là biến ngẫu nhiên hình học với xác suất thành công `π/4`, kỳ vọng `4/π ≈ 1.273` lần. Vòng lặp **kết thúc với xác suất 1** (không có ngưỡng cứng, nhưng xác suất cần quá `k` lần giảm như `(1 - π/4)^k`).

In [12]:
class SolReject:
    def __init__(self, radius: float, x_center: float, y_center: float):
        self.radius = radius
        self.x_center = x_center
        self.y_center = y_center

    def randPoint(self) -> List[float]:
        R, xc, yc = self.radius, self.x_center, self.y_center
        while True:
            x = random.uniform(xc - R, xc + R)
            y = random.uniform(yc - R, yc + R)
            if (x - xc) ** 2 + (y - yc) ** 2 <= R * R:
                return [x, y]

random.seed(478)
print(chay(SolReject, ops, args))

[None, [-0.21822544455115378, -0.8667590831946241], [0.11424595632224133, 0.7678320901272775], [0.7532024582481518, 0.11791454239585653]]


**Code nộp LeetCode (Cách 2):**

```python
class Solution:
    def __init__(self, radius: float, x_center: float, y_center: float):
        self.radius = radius
        self.x_center = x_center
        self.y_center = y_center

    def randPoint(self) -> List[float]:
        R, xc, yc = self.radius, self.x_center, self.y_center
        while True:
            x = random.uniform(xc - R, xc + R)
            y = random.uniform(yc - R, yc + R)
            if (x - xc) ** 2 + (y - yc) ** 2 <= R * R:
                return [x, y]
```

**Độ phức tạp:** Time kỳ vọng `O(1)` (trung bình `4/π ≈ 1.27` lần thử), Space `O(1)`. Tệ nhất không bị chặn (nhưng xác suất cực nhỏ).

### Cách 3 — Tọa độ cực với căn bậc hai (cách nộp)

**Giải thích từng đoạn:**

1. `u = random.random()` đều trong `[0, 1)`; `r = R * sqrt(u)`: để `P(r <= t R) = t^2`.
2. `theta = random.random() * 2π`: góc đều.
3. `x = xc + r cos θ`, `y = yc + r sin θ`.

Số lần gọi bộ sinh cố định `2`, không có vòng lặp: thời gian `O(1)` tất định. Dạng viết tương đương trong lời giải chuẩn: `length = sqrt(uniform(0, R²))`.

In [13]:
class SolPolar:
    def __init__(self, radius: float, x_center: float, y_center: float):
        self.radius = radius
        self.x_center = x_center
        self.y_center = y_center

    def randPoint(self) -> List[float]:
        r = self.radius * math.sqrt(random.random())          # căn bậc hai -> đều theo diện tích
        theta = random.random() * 2 * math.pi
        return [self.x_center + r * math.cos(theta), self.y_center + r * math.sin(theta)]

random.seed(478)
print(chay(SolPolar, ops, args))

[None, [0.5090756424490186, -0.8396389514914637], [0.3361235672650776, 0.04552803629751672], [0.5712313678598343, 0.2541298921760399]]


**Code nộp LeetCode (Cách 3):**

```python
import math
import random

class Solution:
    def __init__(self, radius: float, x_center: float, y_center: float):
        self.radius = radius
        self.x_center = x_center
        self.y_center = y_center

    def randPoint(self) -> List[float]:
        r = self.radius * math.sqrt(random.random())
        theta = random.random() * 2 * math.pi
        return [self.x_center + r * math.cos(theta), self.y_center + r * math.sin(theta)]
```

**Độ phức tạp:** Time `O(1)` (đúng 2 lần gọi bộ sinh), Space `O(1)`.

### Cách 4 — Hướng ngẫu nhiên từ phân phối chuẩn (không cần `sin`/`cos`)

**Ý tưởng:** nếu `(g1, g2)` là hai số độc lập theo phân phối chuẩn tắc `N(0, 1)`, thì vector `(g1, g2)` có phân phối **đối xứng quay**, nên `(g1, g2) / sqrt(g1^2 + g2^2)` là một **hướng đều** trên đường tròn đơn vị. Nhân với `r = R sqrt(U)` rồi cộng tâm. Python có sẵn `random.gauss(0, 1)` (hoặc `random.normalvariate`).

**Chú ý:** nếu `g1 = g2 = 0` (cực kỳ hiếm) thì chia cho `0`; ta lặp lại cho an toàn.

In [14]:
class SolGauss:
    def __init__(self, radius: float, x_center: float, y_center: float):
        self.radius = radius
        self.x_center = x_center
        self.y_center = y_center

    def randPoint(self) -> List[float]:
        while True:
            g1, g2 = random.gauss(0, 1), random.gauss(0, 1)
            norm = math.hypot(g1, g2)
            if norm > 0:
                break
        r = self.radius * math.sqrt(random.random())
        return [self.x_center + r * g1 / norm, self.y_center + r * g2 / norm]

random.seed(478)
print(chay(SolGauss, ops, args))

[None, [0.33062494073442295, -0.07575620848794158], [0.2557737439257612, 0.034644629033763315], [-0.8766095645161863, -0.32887216440863165]]


**Code nộp LeetCode (Cách 4):**

```python
import math
import random

class Solution:
    def __init__(self, radius: float, x_center: float, y_center: float):
        self.radius = radius
        self.x_center = x_center
        self.y_center = y_center

    def randPoint(self) -> List[float]:
        while True:
            g1, g2 = random.gauss(0, 1), random.gauss(0, 1)
            norm = math.hypot(g1, g2)
            if norm > 0:
                break
        r = self.radius * math.sqrt(random.random())
        return [self.x_center + r * g1 / norm, self.y_center + r * g2 / norm]
```

**Độ phức tạp:** Time kỳ vọng `O(1)`, Space `O(1)`. Chậm hơn Cách 3 một chút (gọi `gauss` hai lần), nhưng cách này **mở rộng được** lên hình cầu trong `d` chiều (chuẩn hóa vector `d` số chuẩn, nhân bán kính `R * U^(1/d)`).

### Cách 5 — Mẹo `max(U1, U2)` thay cho `sqrt`

**Ý tưởng:** nếu `U1, U2` độc lập đều trong `[0, 1)` thì `P(max(U1, U2) <= t) = P(U1 <= t) P(U2 <= t) = t^2`: **cùng phân phối** với `sqrt(U)`. Do đó `r = R * max(U1, U2)` cũng cho phân bố đều theo diện tích, không cần `sqrt`. Tốn 3 lần gọi bộ sinh thay vì 2, nhưng không cần `sqrt`, và là minh họa đẹp: *cùng một phân phối có thể sinh bằng nhiều cách*.

In [15]:
class SolMax:
    def __init__(self, radius: float, x_center: float, y_center: float):
        self.radius = radius
        self.x_center = x_center
        self.y_center = y_center

    def randPoint(self) -> List[float]:
        r = self.radius * max(random.random(), random.random())      # P(max <= t) = t^2
        theta = random.random() * 2 * math.pi
        return [self.x_center + r * math.cos(theta), self.y_center + r * math.sin(theta)]

random.seed(478)
print(chay(SolMax, ops, args))

[None, [0.723012994324715, 0.6378404789191879], [0.3571394418108456, 0.15888449575036542], [0.6312795349679854, -0.6187032596096038]]


**Code nộp LeetCode (Cách 5):**

```python
import math
import random

class Solution:
    def __init__(self, radius: float, x_center: float, y_center: float):
        self.radius = radius
        self.x_center = x_center
        self.y_center = y_center

    def randPoint(self) -> List[float]:
        r = self.radius * max(random.random(), random.random())
        theta = random.random() * 2 * math.pi
        return [self.x_center + r * math.cos(theta), self.y_center + r * math.sin(theta)]
```

**Độ phức tạp:** Time `O(1)` (3 lần gọi bộ sinh), Space `O(1)`.

## 8. TRACE từng bước

Để trace được một thuật toán ngẫu nhiên, ta **thay bộ sinh thật bằng bộ sinh "kịch bản"**: một đối tượng trả về lần lượt các giá trị ta chọn sẵn. Nhờ vậy xem được từng bước tính toán.

### 8.1. Cách 3 (tọa độ cực) với `u = 0.25`, `θ = 0.25 * 2π = π/2`; `R = 2`, tâm `(1, 1)`

| bước | công thức | giá trị |
|------|-----------|---------|
| 1 | `u` | `0.25` |
| 2 | `r = R * sqrt(u)` | `2 * 0.5 = 1.0` |
| 3 | `v` (cho góc) | `0.25` |
| 4 | `θ = v * 2π` | `π/2` |
| 5 | `x = xc + r cos θ` | `1 + 1.0 * 0 = 1.0` |
| 6 | `y = yc + r sin θ` | `1 + 1.0 * 1 = 2.0` |

In [16]:
class KichBan:
    """Nguồn số 'ngẫu nhiên' theo kịch bản cho trước, dùng để trace."""
    def __init__(self, gia_tri):
        self.gia_tri = list(gia_tri)

    def random(self):
        return self.gia_tri.pop(0)

    def uniform(self, a, b):
        return a + (b - a) * self.random()

def trace_polar(rng, R, xc, yc):
    u = rng.random()
    r = R * math.sqrt(u)
    v = rng.random()
    theta = v * 2 * math.pi
    x = xc + r * math.cos(theta)
    y = yc + r * math.sin(theta)
    print(f"u = {u}  -> r = R * sqrt(u) = {R} * {math.sqrt(u):.4f} = {r:.4f}")
    print(f"v = {v}  -> theta = v * 2pi = {theta:.4f} rad ({math.degrees(theta):.1f} do)")
    print(f"x = {xc} + {r:.4f} * cos(theta) = {x:.4f}")
    print(f"y = {yc} + {r:.4f} * sin(theta) = {y:.4f}")
    return [x, y]

diem = trace_polar(KichBan([0.25, 0.25]), 2.0, 1.0, 1.0)
print("KET QUA:", [round(c, 6) for c in diem])

u = 0.25  -> r = R * sqrt(u) = 2.0 * 0.5000 = 1.0000
v = 0.25  -> theta = v * 2pi = 1.5708 rad (90.0 do)
x = 1.0 + 1.0000 * cos(theta) = 1.0000
y = 1.0 + 1.0000 * sin(theta) = 2.0000
KET QUA: [1.0, 2.0]


### 8.2. Cách 2 (lấy mẫu loại bỏ) với kịch bản có một lần bị loại

`R = 1`, tâm `(0, 0)`. Kịch bản: lần thử 1 cho `(0.95, 0.95)` → tọa độ `(0.9, 0.9)`, `0.81 + 0.81 = 1.62 > 1` → **loại**. Lần thử 2 cho `(0.6, 0.4)` → tọa độ `(0.2, -0.2)`, `0.04 + 0.04 = 0.08 <= 1` → **nhận**.

In [17]:
def trace_reject(rng, R, xc, yc):
    lan = 0
    while True:
        lan += 1
        x = rng.uniform(xc - R, xc + R)
        y = rng.uniform(yc - R, yc + R)
        d2 = (x - xc) ** 2 + (y - yc) ** 2
        ok = d2 <= R * R
        print(f"lan {lan}: x = {x:.3f}, y = {y:.3f}, d^2 = {d2:.3f} {'<=' if ok else '>'} R^2 = {R * R} -> {'NHAN' if ok else 'LOAI, thu lai'}")
        if ok:
            return [x, y]

diem = trace_reject(KichBan([0.95, 0.95, 0.6, 0.4]), 1.0, 0.0, 0.0)
print("KET QUA:", [round(c, 6) for c in diem])

lan 1: x = 0.900, y = 0.900, d^2 = 1.620 > R^2 = 1.0 -> LOAI, thu lai
lan 2: x = 0.200, y = -0.200, d^2 = 0.080 <= R^2 = 1.0 -> NHAN
KET QUA: [0.2, -0.2]


### 8.3. Trace phép đo `χ²` trên một mẫu nhỏ

Để hiểu cách kiểm thử ở mục 12, ta chia mẫu 12 điểm vào `k = 4` vòng diện tích bằng nhau (ngưỡng vòng: `d²/R² ∈ [0, .25), [.25, .5), [.5, .75), [.75, 1]`).

In [18]:
def trace_chi2(diem, R, xc, yc, k=4):
    dem = [0] * k
    for x, y in diem:
        d2 = ((x - xc) ** 2 + (y - yc) ** 2) / (R * R)
        vong = min(k - 1, int(k * d2))
        dem[vong] += 1
        print(f"  diem ({x:6.3f}, {y:6.3f}): d^2/R^2 = {d2:.3f} -> vong {vong}")
    ky_vong = len(diem) / k
    print("so diem moi vong:", dem, "| ky vong moi vong:", ky_vong)
    ct = [f"({o} - {ky_vong:g})^2 / {ky_vong:g} = {(o - ky_vong) ** 2 / ky_vong:.3f}" for o in dem]
    print("so hang chi^2:", ct)
    print("chi^2 =", round(chi_binh_phuong(dem, ky_vong), 3))

random.seed(5)
mau = [SolPolar(1.0, 0.0, 0.0).randPoint() for _ in range(12)]
trace_chi2(mau, 1.0, 0.0, 0.0)

  diem (-0.041, -0.788): d^2/R^2 = 0.623 -> vong 2
  diem ( 0.834, -0.315): d^2/R^2 = 0.795 -> vong 3
  diem ( 0.760, -0.403): d^2/R^2 = 0.740 -> vong 2
  diem (-0.166,  0.037): d^2/R^2 = 0.029 -> vong 0
  diem (-0.576, -0.782): d^2/R^2 = 0.943 -> vong 3
  diem ( 0.719,  0.620): d^2/R^2 = 0.901 -> vong 3
  diem ( 0.015,  0.685): d^2/R^2 = 0.469 -> vong 1
  diem (-0.659, -0.330): d^2/R^2 = 0.544 -> vong 2
  diem ( 0.024,  0.112): d^2/R^2 = 0.013 -> vong 0
  diem ( 0.457, -0.265): d^2/R^2 = 0.279 -> vong 1
  diem ( 0.471,  0.738): d^2/R^2 = 0.766 -> vong 3
  diem ( 0.574,  0.684): d^2/R^2 = 0.797 -> vong 3
so diem moi vong: [2, 2, 3, 5] | ky vong moi vong: 3.0
so hang chi^2: ['(2 - 3)^2 / 3 = 0.333', '(2 - 3)^2 / 3 = 0.333', '(3 - 3)^2 / 3 = 0.000', '(5 - 3)^2 / 3 = 1.333']
chi^2 = 2.0


## 9. Code nộp LeetCode

Ta nộp **Cách 3** (tọa độ cực với `sqrt`): ngắn, thời gian tất định, đúng phân bố. LeetCode đã `import random, math` sẵn; trong notebook ta đã nhập chúng ở trên.

```python
class Solution:
    def __init__(self, radius: float, x_center: float, y_center: float):
        self.radius = radius
        self.x_center = x_center
        self.y_center = y_center

    def randPoint(self) -> List[float]:
        r = self.radius * math.sqrt(random.random())
        theta = random.random() * 2 * math.pi
        return [self.x_center + r * math.cos(theta), self.y_center + r * math.sin(theta)]
```

In [19]:
class Solution:
    def __init__(self, radius: float, x_center: float, y_center: float):
        self.radius = radius
        self.x_center = x_center
        self.y_center = y_center

    def randPoint(self) -> List[float]:
        r = self.radius * math.sqrt(random.random())
        theta = random.random() * 2 * math.pi
        return [self.x_center + r * math.cos(theta), self.y_center + r * math.sin(theta)]

random.seed(478)
for p in chay(Solution, ops, args):
    print(p)

None
[0.5090756424490186, -0.8396389514914637]
[0.3361235672650776, 0.04552803629751672]
[0.5712313678598343, 0.2541298921760399]


## 10. Vì sao thuật toán đúng

**Yêu cầu:** với mọi vùng con `A` của hình tròn `D`, `P(điểm ∈ A) = diện tích(A) / diện tích(D)`.

**Cách 3.** Dùng tọa độ cực. Mật độ xác suất đều trên hình tròn theo tọa độ `(r, θ)` là `f(r, θ) = (1 / π R^2) * r` (phần tử diện tích `dA = r dr dθ`). Tách `f(r, θ) = f_r(r) f_θ(θ)` với `f_θ(θ) = 1 / 2π` (góc đều, độc lập) và `f_r(r) = 2 r / R^2` trên `[0, R]`. Hàm phân phối của `r` là `F_r(t) = ∫_0^t 2 s / R^2 ds = t^2 / R^2`. Theo phương pháp **biến đổi nghịch**, nếu `U` đều trong `[0, 1)` thì `r = F_r^{-1}(U) = R sqrt(U)` có đúng hàm phân phối `F_r`. Vậy `(r, θ)` có đúng mật độ cần, tức điểm đều trong hình tròn.

**Cách 2.** Điểm `(x, y)` đều trong hình vuông `S ⊇ D`. Với `A ⊆ D`: `P(điểm ∈ A | điểm ∈ D) = P(điểm ∈ A) / P(điểm ∈ D) = (area(A)/area(S)) / (area(D)/area(S)) = area(A)/area(D)`. Vậy điều kiện hóa trên `D` cho đúng phân bố đều trên `D`. Vòng lặp dừng với xác suất `1` vì mỗi lần thành công với xác suất `π/4 > 0`.

**Cách 4.** `(g1, g2)` độc lập chuẩn tắc có mật độ `(1 / 2π) e^{-(g1^2 + g2^2)/2}` chỉ phụ thuộc `g1^2 + g2^2`, nên đối xứng quay; hướng `(g1, g2) / ||(g1, g2)||` đều trên đường tròn, độc lập với độ dài. Bán kính lấy như Cách 3.

**Cách 5.** `P(max(U1, U2) <= t) = P(U1 <= t, U2 <= t) = t^2`: đúng bằng `F_r(R t) = t^2`. Hai biến có cùng phân phối nên thay thế được cho `sqrt(U)`.

**Điểm "trong hình tròn":** với Cách 3, `r <= R` vì `sqrt(U) < 1`; khoảng cách từ điểm tới tâm là `r` (lý thuyết), tính số thực có thể lệch cỡ `10^-16` tương đối, nên test dùng dung sai nhỏ (mục 12).

## 11. Phân tích Time / Space Complexity

| Cách | Time mỗi lần `randPoint` | Space | Ghi chú |
|------|--------------------------|-------|---------|
| 1. Bán kính đều (SAI) | `O(1)` | `O(1)` | không đúng phân bố |
| 2. Loại bỏ | `O(1)` kỳ vọng (`4/π ≈ 1.27` lần thử) | `O(1)` | không có chặn tệ nhất |
| 3. Cực + `sqrt` (nộp) | `O(1)` tất định, 2 lần gọi RNG | `O(1)` | ngắn, nhanh |
| 4. Hướng chuẩn hóa từ Gauss | `O(1)` kỳ vọng | `O(1)` | mở rộng được lên nhiều chiều |
| 5. `max(U1, U2)` | `O(1)` tất định, 3 lần gọi RNG | `O(1)` | không cần `sqrt` |

Khởi tạo `__init__`: `O(1)`. Với tối đa `3 × 10^4` lần gọi, tổng chi phí chỉ vài chục mili-giây.

## 12. Bộ test

Với bài **ngẫu nhiên**, ta không so đáp án cứng mà kiểm tra:

1. **Hình dạng kết quả:** mỗi lần gọi trả về list gồm đúng 2 số thực.
2. **Thuộc hình tròn:** khoảng cách tới tâm `<= R` (dung sai làm tròn nhỏ).
3. **Phân bố đều:** `χ²` theo **vòng diện tích bằng nhau** (kiểm tra bán kính) và theo **cung góc bằng nhau** (kiểm tra góc); thêm kiểm tra trung bình `x`, `y` về tâm và `E[d²] = R²/2`.

Tất cả dùng **seed cố định** nên kết quả tái lập được.

In [20]:
NGUONG_9 = 27.877      # chi-square, 9 bậc tự do, mức 0.001 (10 vòng)
NGUONG_7 = 24.322      # chi-square, 7 bậc tự do, mức 0.001 (8 cung)

def chi2_cung(diem, xc, yc, s=8):
    dem = [0] * s
    for x, y in diem:
        goc = math.atan2(y - yc, x - xc) % (2 * math.pi)
        dem[min(s - 1, int(goc / (2 * math.pi) * s))] += 1
    return chi_binh_phuong(dem, len(diem) / s), dem

def kiem_tra_phan_bo(diem, R, xc, yc, in_ra=True):
    """Trả về dict thống kê và assert các tính chất của phân bố đều trong hình tròn."""
    N = len(diem)
    assert all(isinstance(p, list) and len(p) == 2 for p in diem)
    assert all(trong_hinh_tron(p, R, xc, yc) for p in diem), "co diem ngoai hinh tron"
    c_vong, _ = chi2_vong(diem, R, xc, yc, 10)
    c_cung, _ = chi2_cung(diem, xc, yc, 8)
    mx = sum(p[0] for p in diem) / N
    my = sum(p[1] for p in diem) / N
    md2 = sum(((p[0] - xc) ** 2 + (p[1] - yc) ** 2) for p in diem) / N / (R * R)
    se_xy = R / (2 * math.sqrt(N))                          # sai số chuẩn của trung bình x (hoặc y)
    se_d2 = math.sqrt(1 / 12) / math.sqrt(N)                # sai số chuẩn của d^2 / R^2
    assert c_vong < NGUONG_9, ("khong deu theo ban kinh", c_vong)
    assert c_cung < NGUONG_7, ("khong deu theo goc", c_cung)
    assert abs(mx - xc) < 4 * se_xy and abs(my - yc) < 4 * se_xy, ("lech tam", mx - xc, my - yc)
    assert abs(md2 - 0.5) < 4 * se_d2, ("E[d^2/R^2] lech", md2)
    if in_ra:
        print(f"  chi2 vong = {c_vong:6.2f} (< {NGUONG_9}), chi2 cung = {c_cung:5.2f} (< {NGUONG_7}), "
              f"lech tam = ({(mx - xc) / se_xy:+.2f}, {(my - yc) / se_xy:+.2f}) se, E[d2/R2] = {md2:.4f}")
    return c_vong, c_cung

cac_cach = {
    "reject": SolReject,
    "polar":  SolPolar,
    "gauss":  SolGauss,
    "max":    SolMax,
    "Solution": Solution,
}

random.seed(478)
for ten, cls in cac_cach.items():
    ob = cls(1.0, 0.0, 0.0)
    diem = [ob.randPoint() for _ in range(30000)]
    print(f"{ten:9}:", end="")
    kiem_tra_phan_bo(diem, 1.0, 0.0, 0.0)
print("Cac cach DUNG deu qua kiem tra phan bo (N = 30000)")

reject   :

  chi2 vong =  10.27 (< 27.877), chi2 cung =  7.65 (< 24.322), lech tam = (+0.57, -0.58) se, E[d2/R2] = 0.5005


polar    :

  chi2 vong =   4.85 (< 27.877), chi2 cung =  2.99 (< 24.322), lech tam = (+0.98, -0.91) se, E[d2/R2] = 0.5007


gauss    :

  chi2 vong =  13.77 (< 27.877), chi2 cung =  6.61 (< 24.322), lech tam = (-0.06, +2.00) se, E[d2/R2] = 0.4993


max      :  chi2 vong =   3.93 (< 27.877), chi2 cung =  6.98 (< 24.322), lech tam = (-0.31, -0.36) se, E[d2/R2] = 0.4992


Solution :

  chi2 vong =   4.34 (< 27.877), chi2 cung =  3.90 (< 24.322), lech tam = (+0.00, +1.98) se, E[d2/R2] = 0.5011
Cac cach DUNG deu qua kiem tra phan bo (N = 30000)


In [21]:
# Cách 1 (bán kính đều) phải THẤT BẠI bài kiểm tra thống kê: kiểm tra test của ta đủ nhạy
random.seed(478)
ob = SolNaive(1.0, 0.0, 0.0)
diem = [ob.randPoint() for _ in range(30000)]
try:
    kiem_tra_phan_bo(diem, 1.0, 0.0, 0.0)
    print("LOI: test khong phat hien cach sai!")
except AssertionError as e:
    print("Test phat hien dung cach sai:", e)
assert trong_hinh_tron(diem[0], 1.0, 0.0, 0.0)       # nhưng điểm vẫn trong hình tròn

Test phat hien dung cach sai: ('khong deu theo ban kinh', 16680.050666666666)


In [22]:
# Nhiều cấu hình: bán kính / tâm khác nhau, tâm lệch, bán kính nhỏ và rất lớn (ràng buộc đề)
cau_hinh = [
    (1.0, 0.0, 0.0),
    (10.0, 5.0, -3.0),
    (0.001, 1.0, 1.0),                     # bán kính rất nhỏ
    (1e8, 1e7, -1e7),                      # bán kính và tâm lớn nhất theo đề
    (5.5, -1e7, 1e7),
    (123.456, 0.0, 9999999.0),
    (1e-3, 1e7, -1e7),                     # bán kính rất nhỏ, tâm rất xa
]
random.seed(4780)
for R, xc, yc in cau_hinh:
    ob = Solution(R, xc, yc)
    diem = [ob.randPoint() for _ in range(20000)]
    print(f"R = {R:<10g} tam = ({xc:g}, {yc:g}):", end="")
    kiem_tra_phan_bo(diem, R, xc, yc)
print("Moi cau hinh deu OK")

R = 1          tam = (0, 0):

  chi2 vong =   7.89 (< 27.877), chi2 cung =  4.36 (< 24.322), lech tam = (-0.03, +1.69) se, E[d2/R2] = 0.4974
R = 10         tam = (5, -3):

  chi2 vong =  11.43 (< 27.877), chi2 cung =  5.76 (< 24.322), lech tam = (-0.76, -0.45) se, E[d2/R2] = 0.4997
R = 0.001      tam = (1, 1):  chi2 vong =   6.11 (< 27.877), chi2 cung =  6.08 (< 24.322), lech tam = (-1.69, +0.28) se, E[d2/R2] = 0.4975
R = 1e+08      tam = (1e+07, -1e+07):  chi2 vong =  12.20 (< 27.877), chi2 cung =  4.23 (< 24.322), lech tam = (+0.25, +2.02) se, E[d2/R2] = 0.4999


R = 5.5        tam = (-1e+07, 1e+07):  chi2 vong =  11.09 (< 27.877), chi2 cung =  5.47 (< 24.322), lech tam = (+1.15, -0.87) se, E[d2/R2] = 0.5038


R = 123.456    tam = (0, 1e+07):

  chi2 vong =   4.87 (< 27.877), chi2 cung =  9.09 (< 24.322), lech tam = (+1.69, -0.69) se, E[d2/R2] = 0.4997
R = 0.001      tam = (1e+07, -1e+07):

  chi2 vong =   4.36 (< 27.877), chi2 cung =  9.75 (< 24.322), lech tam = (+0.36, +0.89) se, E[d2/R2] = 0.4996
Moi cau hinh deu OK


In [23]:
# Mọi cách trên cùng cấu hình lệch tâm; và một lượt chạy bằng bộ chạy chuỗi lệnh kiểu LeetCode
random.seed(4781)
ops = ["Solution"] + ["randPoint"] * 20000
args = [[2.5, -3.0, 4.0]] + [[]] * 20000
for ten, cls in cac_cach.items():
    ket_qua = chay(cls, ops, args)
    assert ket_qua[0] is None
    diem = ket_qua[1:]
    print(f"{ten:9}:", end="")
    kiem_tra_phan_bo(diem, 2.5, -3.0, 4.0)

reject   :  chi2 vong =   7.10 (< 27.877), chi2 cung =  9.61 (< 24.322), lech tam = (-1.06, -0.70) se, E[d2/R2] = 0.4997
polar    :  chi2 vong =   7.03 (< 27.877), chi2 cung =  9.21 (< 24.322), lech tam = (+0.35, +2.02) se, E[d2/R2] = 0.5028
gauss    :

  chi2 vong =   2.48 (< 27.877), chi2 cung =  5.24 (< 24.322), lech tam = (+1.38, +0.25) se, E[d2/R2] = 0.5005
max      :

  chi2 vong =  11.82 (< 27.877), chi2 cung =  9.06 (< 24.322), lech tam = (-1.69, -0.41) se, E[d2/R2] = 0.4972
Solution :  chi2 vong =  10.29 (< 27.877), chi2 cung =  7.64 (< 24.322), lech tam = (+0.97, -0.41) se, E[d2/R2] = 0.5007


In [24]:
# Nguồn ngẫu nhiên khác: bộ sinh LCG tự dựng cũng cho phân bố đều (thuật toán không phụ thuộc bộ sinh)
def diem_cuc(rng, R, xc, yc):
    r = R * math.sqrt(rng.random())
    theta = rng.random() * 2 * math.pi
    return [xc + r * math.cos(theta), yc + r * math.sin(theta)]

g = LCG(20240478)
diem = [diem_cuc(g, 3.0, 1.0, -1.0) for _ in range(30000)]
print("LCG tu dung:", end="")
kiem_tra_phan_bo(diem, 3.0, 1.0, -1.0)

# Tái lập: cùng seed -> cùng chuỗi điểm; seed khác -> khác
random.seed(99)
a = [Solution(1.0, 0.0, 0.0).randPoint() for _ in range(5)]
random.seed(99)
b = [Solution(1.0, 0.0, 0.0).randPoint() for _ in range(5)]
random.seed(100)
c = [Solution(1.0, 0.0, 0.0).randPoint() for _ in range(5)]
assert a == b and a != c
print("Tai lap theo seed OK")

LCG tu dung:  chi2 vong =  11.92 (< 27.877), chi2 cung =  5.42 (< 24.322), lech tam = (-0.02, +0.61) se, E[d2/R2] = 0.4978
Tai lap theo seed OK


In [25]:
# Kiểm tra cho Cách 2: số lần thử trung bình ~ 4/pi, và các điểm nằm đúng hình vuông bao
def dem_lan_thu(R, N, seed):
    rng = random.Random(seed)
    tong = 0
    for _ in range(N):
        while True:
            tong += 1
            x, y = rng.uniform(-R, R), rng.uniform(-R, R)
            if x * x + y * y <= R * R:
                break
    return tong / N

tb = dem_lan_thu(1.0, 100000, 2)
print(f"so lan thu trung binh = {tb:.4f}, ly thuyet 4/pi = {4 / math.pi:.4f}")
assert abs(tb - 4 / math.pi) < 0.02

# Hiệu năng: 3 * 10^4 lần gọi cho từng cách
random.seed(1)
for ten, cls in cac_cach.items():
    ob = cls(1e8, 1e7, -1e7)
    t0 = time.perf_counter()
    for _ in range(30000):
        ob.randPoint()
    print(f"{ten:9}: 30000 lan goi trong {time.perf_counter() - t0:.3f}s")

so lan thu trung binh = 1.2676, ly thuyet 4/pi = 1.2732
reject   : 30000 lan goi trong 0.025s
polar    : 30000 lan goi trong 0.008s


gauss    : 30000 lan goi trong 0.034s
max      : 30000 lan goi trong 0.016s


Solution : 30000 lan goi trong 0.011s


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Bán kính đều thay vì lấy căn bậc hai (điểm dồn về tâm)

Đã thấy ở mục 5: `r = uniform(0, R)` cho `χ²` rất lớn. **Code đúng:** `r = R * sqrt(random())`.

### Lỗi 2 — Lấy mẫu loại bỏ nhưng quên cộng tâm khi kiểm tra / khi tạo điểm

Một lỗi hay gặp khi `tâm ≠ (0, 0)`: kiểm tra `x*x + y*y <= R*R` trên tọa độ **tuyệt đối** thay vì tương đối so với tâm.

In [26]:
def rand_point_sai_tam(R, xc, yc, toi_da_thu=1000):
    """Phiên bản SAI (có giới hạn số lần thử để không treo): kiểm tra trên tọa độ tuyệt đối."""
    for lan in range(1, toi_da_thu + 1):
        x = random.uniform(xc - R, xc + R)
        y = random.uniform(yc - R, yc + R)
        if x * x + y * y <= R * R:                          # SAI: phải trừ tâm
            return [x, y], lan
    return None, toi_da_thu

random.seed(13)
diem, so_lan = rand_point_sai_tam(2.0, 10.0, 10.0)
print("ket qua:", diem, "| so lan thu:", so_lan)

ket qua: None | so lan thu: 1000


Với tâm `(10, 10)` và `R = 2`, điều kiện `x² + y² <= 4` **không bao giờ** đúng (vì `x, y >= 8`), nên vòng `while True` **không kết thúc**. Ta đã giới hạn số lần thử (1000) ở trên để cell không bị treo, và thấy không có điểm nào được nhận. Nguyên nhân bằng tính toán: điểm gần gốc tọa độ nhất trong hình vuông bao là `(8, 8)`.

In [27]:
xc = yc = 10.0
R = 2.0
x_min, y_min = xc - R, yc - R
print("diem gan goc nhat trong hinh vuong bao:", (x_min, y_min), "-> x^2 + y^2 =", x_min ** 2 + y_min ** 2, "> R^2 =", R * R)
print("=> dieu kien sai luon False, vong lap vo han (treo / TLE)")

diem gan goc nhat trong hinh vuong bao: (8.0, 8.0) -> x^2 + y^2 = 128.0 > R^2 = 4.0
=> dieu kien sai luon False, vong lap vo han (treo / TLE)


**Vì sao sai:** hình tròn nằm quanh tâm, không quanh gốc tọa độ. **Code đúng:** `(x - xc)**2 + (y - yc)**2 <= R*R` (và `[xc + r cos θ, yc + r sin θ]` ở cách cực).

### Lỗi 3 — Khoảng góc chỉ `[0, π]` (hoặc đơn vị độ thay vì radian)

Nếu quên `2π` (ví dụ `theta = random.random() * math.pi`), các điểm chỉ nằm ở nửa trên của hình tròn.

In [28]:
class SolNuaGoc:
    def __init__(self, radius: float, x_center: float, y_center: float):
        self.radius = radius
        self.x_center = x_center
        self.y_center = y_center

    def randPoint(self) -> List[float]:
        r = self.radius * math.sqrt(random.random())
        theta = random.random() * math.pi                 # SAI: thiếu hệ số 2
        return [self.x_center + r * math.cos(theta), self.y_center + r * math.sin(theta)]

random.seed(478)
ob = SolNuaGoc(1.0, 0.0, 0.0)
diem = [ob.randPoint() for _ in range(30000)]
assert all(trong_hinh_tron(p, 1.0, 0.0, 0.0) for p in diem)        # vẫn trong hình tròn
c_cung, dem = chi2_cung(diem, 0.0, 0.0, 8)
print("so diem moi cung goc:", dem)
print("chi2 cung =", round(c_cung, 1), "> nguong", NGUONG_7, "->", "KHONG DEU" if c_cung > NGUONG_7 else "deu")

so diem moi cung goc: [7510, 7512, 7477, 7501, 0, 0, 0, 0]
chi2 cung = 30000.2 > nguong 24.322 -> KHONG DEU


**Vì sao sai:** góc chỉ phủ nửa vòng, nên `y >= yc` luôn đúng: nửa dưới trống. Tính chất "nằm trong hình tròn" vẫn đúng, nên chỉ test **phân bố** mới lộ lỗi.

**Code đúng:** `theta = random.random() * 2 * math.pi`.

### Lỗi 4 — `sqrt(uniform(0, R))` thay vì `sqrt(uniform(0, R²))` (sai thang đo)

Lời giải chuẩn viết `sqrt(uniform(0, R²))`. Nếu quên bình phương `R`, bán kính tối đa chỉ là `sqrt(R)`.

In [29]:
class SolSaiThang:
    def __init__(self, radius: float, x_center: float, y_center: float):
        self.radius = radius
        self.x_center = x_center
        self.y_center = y_center

    def randPoint(self) -> List[float]:
        r = math.sqrt(random.uniform(0, self.radius))        # SAI: phải là uniform(0, radius**2)
        theta = random.random() * 2 * math.pi
        return [self.x_center + r * math.cos(theta), self.y_center + r * math.sin(theta)]

random.seed(478)
ob = SolSaiThang(9.0, 0.0, 0.0)
diem = [ob.randPoint() for _ in range(30000)]
khoang_cach_max = max(math.hypot(p[0], p[1]) for p in diem)
print("R = 9 nhung khoang cach lon nhat chi ~", round(khoang_cach_max, 3), "(ly thuyet sqrt(9) = 3)")
c_vong, dem = chi2_vong(diem, 9.0, 0.0, 0.0, 10)
print("so diem moi vong:", dem)
print("chi2 vong =", round(c_vong, 1), "> nguong", NGUONG_9)

R = 9 nhung khoang cach lon nhat chi ~ 3.0 (ly thuyet sqrt(9) = 3)


so diem moi vong: [27002, 2998, 0, 0, 0, 0, 0, 0, 0, 0]
chi2 vong = 216032.0 > nguong 27.877


**Vì sao sai:** với `R > 1` các điểm không bao giờ chạm vành ngoài (tất cả dồn trong bán kính `sqrt(R)`); với `R < 1` thì `sqrt(R) > R`, điểm có thể **vượt ra ngoài** hình tròn! Đây là lý do cần test "thuộc hình tròn" **và** test phân bố.

**Code đúng:** `r = math.sqrt(random.uniform(0, R * R))` hoặc `R * math.sqrt(random.random())`.

### Lỗi 5 — Đặt lại seed trong `randPoint` (hoặc trong `__init__` mỗi lần)

Gọi `random.seed(...)` **bên trong** `randPoint` làm mọi lần gọi trả về **cùng một điểm**.

In [30]:
class SolSeedMoiLan:
    def __init__(self, radius: float, x_center: float, y_center: float):
        self.radius = radius
        self.x_center = x_center
        self.y_center = y_center

    def randPoint(self) -> List[float]:
        random.seed(42)                                        # SAI: đặt lại seed mỗi lần gọi
        r = self.radius * math.sqrt(random.random())
        theta = random.random() * 2 * math.pi
        return [self.x_center + r * math.cos(theta), self.y_center + r * math.sin(theta)]

ob = SolSeedMoiLan(1.0, 0.0, 0.0)
diem = [ob.randPoint() for _ in range(5)]
print(diem)
print("so diem khac nhau:", len({tuple(p) for p in diem}), "/ 5")

[[0.7897882977934517, 0.12514488853487296], [0.7897882977934517, 0.12514488853487296], [0.7897882977934517, 0.12514488853487296], [0.7897882977934517, 0.12514488853487296], [0.7897882977934517, 0.12514488853487296]]
so diem khac nhau: 1 / 5


**Vì sao sai:** seed đặt lại → bộ sinh khởi động lại từ cùng trạng thái → cùng điểm. Chỉ đặt seed **một lần**, **ở bên ngoài** (trong test), không đặt trong lớp nộp bài.

### Lỗi 6 (kỹ thuật kiểm thử) — So sánh đầu ra với ví dụ của đề

Ví dụ của đề cho `[-0.02493, -0.38077]` ở lần gọi đầu; code đúng của bạn gần như chắc chắn cho điểm **khác** (vì ngẫu nhiên). Với bài ngẫu nhiên, hãy kiểm tra **tính chất** (trong hình tròn, đều), không so đáp án cứng.

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu:** "chọn ngẫu nhiên **đều** trong miền hình học" → hoặc **loại bỏ** (rejection sampling trong hộp bao), hoặc **biến đổi nghịch** (tìm hàm phân phối rồi nghịch đảo).
- **Hình tròn:** `r = R * sqrt(U)`, không phải `R * U`. Hình cầu 3D: `r = R * U^(1/3)`; `d` chiều: `R * U^(1/d)`.
- **Lấy mẫu loại bỏ:** hiệu quả khi tỷ lệ diện tích `miền / hộp bao` không quá nhỏ (ở đây `π/4 ≈ 78,5%`). Với miền mỏng/xa hộp bao thì tỉ lệ thấp, loại bỏ trở nên tốn.
- **Kiểm thử ngẫu nhiên:** cố định **seed**; kiểm tra **tính chất** (miền) và **thống kê** (χ² trên các ô diện tích bằng nhau); chọn ngưỡng ở mức ý nghĩa thấp (0,001) để hiếm khi báo nhầm.
- **Hiệu chỉnh số học:** bán kính tới `10^8`, tâm tới `10^7`: dung sai khi kiểm tra "trong hình tròn" nên **tương đối** theo độ lớn.
- **Bài họ hàng:** LeetCode 497 *Random Point in Non-overlapping Rectangles*, 519 *Random Flip Matrix*, 528 *Random Pick with Weight*, 382 *Linked List Random Node*, 384 *Shuffle an Array*.

## 15. Biến thể

1. **Điểm đều trong hình vành khuyên** (giữa bán kính `r1` và `r2`): `r = sqrt(U * (r2² - r1²) + r1²)`.
2. **Điểm đều trong hình cầu / quả cầu `d` chiều:** hướng chuẩn hóa từ `d` số chuẩn, bán kính `R * U^(1/d)`.
3. **Điểm đều trên đường tròn (chỉ viền):** `r = R`, góc đều.
4. **Điểm đều trong tam giác:** chọn `u, v` đều, nếu `u + v > 1` thì phản chiếu `(1 - u, 1 - v)`, rồi nội suy trọng tâm.
5. **Phân bố không đều có chủ đích** (ví dụ mật độ giảm dần theo bán kính): đổi hàm phân phối `F_r`, nghịch đảo tương ứng.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time/lần gọi | Space | Khi nào dùng |
|------|---------|--------------|-------|--------------|
| 1. Bán kính đều | `r = uniform(0, R)` | `O(1)` | `O(1)` | **sai** (chỉ để học) |
| 2. Loại bỏ | điểm đều trong hộp bao, giữ nếu trong tròn | `O(1)` kỳ vọng | `O(1)` | không cần công thức; miền phức tạp |
| 3. Cực + `sqrt` | biến đổi nghịch `r = R sqrt(U)` | `O(1)` tất định | `O(1)` | **nộp bài**, nhanh và gọn |
| 4. Hướng từ Gauss | chuẩn hóa vector chuẩn + `sqrt(U)` | `O(1)` kỳ vọng | `O(1)` | mở rộng nhiều chiều |
| 5. `max(U1, U2)` | cùng phân phối với `sqrt(U)` | `O(1)` tất định | `O(1)` | khi tránh `sqrt`, minh họa phân phối |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: (a) `math.sqrt(0.25)`; (b) `round(4 / math.pi, 3)`; (c) điểm `(1, 1)` có nằm trong hình tròn bán kính `1` tâm gốc không? (d) nếu `u = 1.0` thì `R * sqrt(u)` với `R = 3` bằng bao nhiêu? (e) xác suất (lý thuyết) để một điểm đều trong hình tròn bán kính `R` có khoảng cách tới tâm `<= R/2`?

**Bài 2 (tính tay).** Với `R = 2`, tâm `(1, 1)`, `u = 0.25`, `v = 0.5` (góc `θ = v * 2π`): tính `r`, `θ`, và điểm `(x, y)`. (Gợi ý: `cos π = -1`, `sin π = 0`.)

**Bài 3 (điền TODO).** Hoàn thành `trong_tron(x, y, R, xc, yc)` trả về `True` nếu điểm `(x, y)` nằm trong hoặc trên hình tròn.

In [31]:
def trong_tron(x, y, R, xc, yc):
    # TODO: trả về (x - xc)**2 + (y - yc)**2 <= R * R
    return None

# KHÔNG gọi trong_tron ở đây để "Run All" không lỗi (chưa điền xong).

**Bài 4 (tự viết lại).** Viết lớp `SolRejectTodo` theo Cách 2 (loại bỏ) **không nhìn lại** mục 7, rồi kiểm tra bằng `kiem_tra_phan_bo`.

**Bài 5 (biến thể / mini challenge).** Viết `rand_vanh_khuyen(r1, r2, xc, yc)` trả về điểm **đều** trong hình vành khuyên `r1 <= d <= r2`. Kiểm tra: mọi điểm có `r1 <= d <= r2` và `χ²` theo các vòng **diện tích bằng nhau** (trong vành khuyên) nhỏ hơn ngưỡng. Gợi ý: `r = sqrt(u * (r2² - r1²) + r1²)`.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: (a) `0.5`; (b) `1.273`; (c) `False` (`1 + 1 = 2 > 1`); (d) `3.0`; (e) `1/4` (tỉ lệ diện tích, **không** phải `1/2`).
- **Mức 2 (vừa):** Bài 2: `r = 2 * sqrt(0.25) = 1.0`, `θ = π`, `x = 1 + 1.0 * (-1) = 0`, `y = 1 + 1.0 * 0 = 1`; điểm `(0, 1)`. Bài 4: nhớ trừ tâm khi kiểm tra.
- **Mức 3 (sâu):** Bài 5: chia vành khuyên thành `k` vòng diện tích bằng nhau: biên `b_i = sqrt(r1² + i (r2² - r1²) / k)`; chỉ số vòng của điểm cách tâm `d`: `floor(k * (d² - r1²) / (r2² - r1²))`.

## 19. Đáp án tham khảo cho bài tập

In [32]:
# Bài 1
assert math.sqrt(0.25) == 0.5
assert round(4 / math.pi, 3) == 1.273
assert (1 ** 2 + 1 ** 2 <= 1) is False
assert 3 * math.sqrt(1.0) == 3.0
random.seed(3)
ty_le = sum(1 for _ in range(40000) if math.hypot(*Solution(1.0, 0.0, 0.0).randPoint()) <= 0.5) / 40000
assert abs(ty_le - 0.25) < 0.01
print("Bài 1 OK, ty le thuc nghiem P(d <= R/2) =", round(ty_le, 3))

# Bài 2
R, xc, yc, u, v = 2.0, 1.0, 1.0, 0.25, 0.5
r = R * math.sqrt(u)
theta = v * 2 * math.pi
x, y = xc + r * math.cos(theta), yc + r * math.sin(theta)
print("Bài 2:", r, round(theta, 4), (round(x, 6), round(y, 6)))
assert r == 1.0 and abs(theta - math.pi) < 1e-12
assert abs(x - 0.0) < 1e-12 and abs(y - 1.0) < 1e-12
print("Bài 2 OK")

# Bài 3
def trong_tron_ans(x, y, R, xc, yc):
    return (x - xc) ** 2 + (y - yc) ** 2 <= R * R

assert trong_tron_ans(0, 0, 1, 0, 0) and trong_tron_ans(1, 0, 1, 0, 0)       # trên đường tròn tính là trong
assert not trong_tron_ans(1, 1, 1, 0, 0)
assert trong_tron_ans(10.5, 10, 1, 10, 10) and not trong_tron_ans(0, 0, 1, 10, 10)
print("Bài 3 OK")

# Bài 4
class SolRejectTodoAns:
    def __init__(self, radius, x_center, y_center):
        self.R, self.xc, self.yc = radius, x_center, y_center

    def randPoint(self):
        while True:
            x = random.uniform(self.xc - self.R, self.xc + self.R)
            y = random.uniform(self.yc - self.R, self.yc + self.R)
            if trong_tron_ans(x, y, self.R, self.xc, self.yc):
                return [x, y]

random.seed(44)
ob = SolRejectTodoAns(3.0, -2.0, 5.0)
diem = [ob.randPoint() for _ in range(30000)]
kiem_tra_phan_bo(diem, 3.0, -2.0, 5.0, in_ra=False)
print("Bài 4 OK")

# Bài 5
def rand_vanh_khuyen(r1, r2, xc, yc):
    u = random.random()
    r = math.sqrt(u * (r2 * r2 - r1 * r1) + r1 * r1)
    theta = random.random() * 2 * math.pi
    return [xc + r * math.cos(theta), yc + r * math.sin(theta)]

random.seed(45)
r1, r2, xc, yc = 2.0, 5.0, 1.0, -1.0
diem = [rand_vanh_khuyen(r1, r2, xc, yc) for _ in range(30000)]
d = [math.hypot(p[0] - xc, p[1] - yc) for p in diem]
assert all(r1 - 1e-9 <= v <= r2 + 1e-9 for v in d)
k = 10
dem = [0] * k
for v in d:
    dem[min(k - 1, int(k * (v * v - r1 * r1) / (r2 * r2 - r1 * r1)))] += 1
chi2 = chi_binh_phuong(dem, len(diem) / k)
print("Bài 5: so diem moi vong dien tich bang nhau:", dem, "chi2 =", round(chi2, 2))
assert chi2 < NGUONG_9
assert chi2_cung(diem, xc, yc, 8)[0] < NGUONG_7
print("Bài 5 OK")

Bài 1 OK, ty le thuc nghiem P(d <= R/2) = 0.25
Bài 2: 1.0 3.1416 (0.0, 1.0)
Bài 2 OK
Bài 3 OK


Bài 4 OK


Bài 5: so diem moi vong dien tich bang nhau: [2994, 3001, 2981, 3024, 3003, 3016, 3081, 3000, 2967, 2933] chi2 = 4.46


Bài 5 OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template (điểm đều trong hình tròn, tọa độ cực):**

```python
class Solution:
    def __init__(self, radius, x_center, y_center):
        self.radius, self.x_center, self.y_center = radius, x_center, y_center

    def randPoint(self):
        r = self.radius * math.sqrt(random.random())     # căn bậc hai: đều theo diện tích
        theta = random.random() * 2 * math.pi            # góc đều [0, 2pi)
        return [self.x_center + r * math.cos(theta), self.y_center + r * math.sin(theta)]
```

**Template (loại bỏ):**

```python
while True:
    x = random.uniform(xc - R, xc + R)
    y = random.uniform(yc - R, yc + R)
    if (x - xc) ** 2 + (y - yc) ** 2 <= R * R:
        return [x, y]
```

**Checklist:**

- [ ] Bán kính lấy qua **căn bậc hai** (`R * sqrt(U)`), **không** phải `R * U`.
- [ ] Góc phủ đủ `[0, 2π)` và dùng **radian**.
- [ ] Luôn **cộng tâm** `(x_center, y_center)` (hoặc trừ tâm khi kiểm tra trong loại bỏ).
- [ ] Không gọi `random.seed` trong lớp nộp bài (chỉ đặt seed trong test).
- [ ] Test bằng **tính chất**: điểm trong hình tròn; phân bố đều bằng `χ²` trên vòng diện tích bằng nhau + cung góc bằng nhau; seed cố định.
- [ ] Test cả cấu hình cực đoan: `R = 10^8`, tâm `±10^7`, `R` rất nhỏ.
- [ ] Đúng signature `class Solution` / `__init__(self, radius, x_center, y_center)` / `randPoint(self) -> List[float]`; mỗi lần gọi `O(1)`.